# Complex scalar Helmholtz fields

The scalar pressure is $p=(1+i)(1+x+2y)$ with unit density and bulk modulus,
frequency $\omega=1/2$ and source $f=-\omega^2p$. The skeleton uses two
interleaved real components for every complex mode. Positive frequency removes
the constant-pressure gauge of pure diffusion.

This low-frequency analytical patch checks complex pressure and gradient
coordinates. It does not certify wave resolution, resonance avoidance or an
absorbing boundary. The acoustic applications and convergence records are in
the numbered Helmholtz notebooks beside this example.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/e280cc3f24cd16aa3aff5cb2e7d1b09e6e318569ba864d59024ee0033d53739a/introductory_methods-companion.zip"
COMPANION_SHA256 = "e280cc3f24cd16aa3aff5cb2e7d1b09e6e318569ba864d59024ee0033d53739a"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("waves/helmholtz/introductory_methods.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)

import numpy as np


## Declare and solve complex Helmholtz equations

The primary native forms use the exact real/imaginary embedding of complex
pressure; no complex coefficient is discarded. Unit density/modulus give
$-\Delta p-\omega^2p=f$ with $\omega=1/2$ and the independent affine
source $f=-\omega^2p$. This chosen finite patch has invertible local operators;
positive frequency alone does not exclude local resonances in other inputs.

$$
\begin{aligned}
a_T(p,v)&=(\nabla p,\nabla v)_T-\omega^2(p,v)_T,\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

The signed trace is $-\nabla p\cdot n$ in two interleaved real components.
The global RHS explicitly prescribes negative complex Dirichlet moments.
The native example has no absorption, PML or resonance-avoidance claim.


In [ ]:
import importlib.util
from pymhm import Equation, LocalEquations, MultiscaleProblem, assemble, columns, rows, compile_form
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.variational_darcy import NativeFormProvider

native_available = importlib.util.find_spec("dolfinx") is not None
if native_available:
    import ufl
    from dolfinx import fem

    mesh = TriangleMesh.unit_square()
    trace_space = SkeletonSpace(mesh, components=2)
    omega = 0.5

    def exact_components(points):
        """Return real/imaginary components of (1+i)(1+x+2y)."""
        scalar = 1 + points[:, 0] + 2 * points[:, 1]
        return np.column_stack((scalar, scalar))

    def local_forms(context):
        """Write both real components of the complex Helmholtz equation."""
        space = fem.functionspace(context.space.mesh, ("Lagrange", 2, (2,)))
        u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
        x = ufl.SpatialCoordinate(context.space.mesh)
        dx = ufl.dx(domain=context.space.mesh)
        exact = ufl.as_vector((1 + x[0] + 2 * x[1], 1 + x[0] + 2 * x[1]))
        return LocalEquations(
            (ufl.inner(ufl.grad(u), ufl.grad(v)) - omega**2 * ufl.inner(u, v)) * dx,
            ufl.inner(-(omega**2) * exact, v) * dx,
            columns(
                *(
                    float(sign) * v[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            rows(
                *(
                    -float(sign) * u[j] * context.ds(side + 1)
                    for side, sign in enumerate(context.signs)
                    for j in range(2)
                )
            ),
            context.trace_dofs,
            metadata={"points": space.tabulate_dof_coordinates()[:, :2].copy()},
        )

    boundary, _ = boundary_data(trace_space, exact_components)
    problem = MultiscaleProblem(
        Equation(0, -boundary),
        NativeFormProvider(mesh, trace_space, local_forms, 2),
        range(len(mesh.cells)),
        trace_space.size,
        (0,) * len(mesh.cells),
    )
    system = assemble(problem)
    solution = system.solve()
    errors = [
        float(np.max(np.abs(field.reshape(-1, 2) - exact_components(record["points"]))))
        for field, record in zip(solution.fields, system.local_metadata, strict=True)
    ]
    assert max(errors) < 1e-10
    print(
        {
            "native_available": True,
            "spaces": "complex P2 pressure / P0 complex trace",
            "maximum_component_nodal_error_by_cell": errors,
            "global_original_relative_residual": solution.raw_residual,
        }
    )
else:
    print(
        {
            "native_available": False,
            "status": "User-written UFL primary solve not executed: select the Pixi fem kernel.",
        }
    )


## Compare the supported methods
These comparisons invoke the existing formulation implementations through the application catalogue and preserve their numerical records. The user-defined primary UFL solve above exercises the generic local/global API; the catalogue drivers remain compatibility comparisons.


In [ ]:
from examples.tutorial_scalar_variants import (
    VARIANTS,
    affine_pressure,
    affine_flux,
    run_variant,
)

record = run_variant("helmholtz")
print(record["spaces_and_conventions"])
print(record["metrics"])


## Plot pressure components and error with the macro mesh

Both components of this affine complex pressure are identical. Sampled nodal values reproduce the affine field; the panel triangulation displays those values separately in each macrocell. The macro mesh is drawn on every panel.


In [ ]:
assert native_available, "Select the locked introduction environment for this UFL demonstration."
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation

fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
for component in range(2):
    for column in range(3):
        ax = axes[component, column]
        for values, record in zip(solution.fields, system.local_metadata, strict=True):
            points = record["points"]
            exact = exact_components(points)[:, component]
            numerical = values.reshape(-1, 2)[:, component]
            shown = (exact, numerical, np.abs(numerical - exact))[column]
            image = ax.tripcolor(Triangulation(points[:, 0], points[:, 1]), shown,
                                 shading="gouraud", vmin=0 if column==2 else 1,
                                 vmax=1e-10 if column==2 else 4)
        ax.triplot(mesh.points[:, 0], mesh.points[:, 1], mesh.cells, color="black", linewidth=0.6)
        name = "Real" if component==0 else "Imaginary"
        ax.set(title=f"{name}: {('exact', 'MHM', 'absolute error')[column]}",
               xlabel="x", ylabel="y", aspect="equal")
        fig.colorbar(image, ax=ax, label="Pressure" if column<2 else "Pressure error")
Path("build/tutorial-methods").mkdir(parents=True, exist_ok=True)
fig.savefig("build/tutorial-methods/helmholtz-affine-fields.png", dpi=200, metadata={"Author":"IPES Research Group", "License":"CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/"})
plt.show()


## Independent conforming Galerkin reference

Assemble the same real/imaginary Helmholtz forms on global P2 grids of 16, 32 and 64 subdivisions. Prescribe the exact exterior pressure strongly and solve with the common sparse factorization owner. The affine field is exactly representable, so refinement verifies reproduction instead of supplying a fitted rate. The MHM example uses only two macrocells.


In [ ]:
from mpi4py import MPI
from dolfinx import mesh as native_mesh
from pymhm.linalg.linear import factorize

classical = []
for resolution in (16, 32, 64):
    domain = native_mesh.create_unit_square(MPI.COMM_SELF, resolution, resolution)
    space = fem.functionspace(domain, ("Lagrange", 2, (2,)))
    u, v = ufl.TrialFunction(space), ufl.TestFunction(space)
    x = ufl.SpatialCoordinate(domain)
    dx = ufl.dx(domain=domain)
    exact = ufl.as_vector((1+x[0]+2*x[1], 1+x[0]+2*x[1]))
    a = (ufl.inner(ufl.grad(u), ufl.grad(v)) - omega**2*ufl.inner(u,v))*dx
    L = -omega**2 * ufl.inner(exact, v)*dx
    points = space.tabulate_dof_coordinates()[:, :2]
    size = 2*len(points)
    matrix, load = compile_form(a, (size,size)), compile_form(L, (size,))
    exterior = np.any(np.isclose(points,0) | np.isclose(points,1), axis=1)
    fixed = (2*np.flatnonzero(exterior)[:,None] + np.arange(2)).ravel()
    free = np.setdiff1d(np.arange(size),fixed)
    values = np.zeros(size)
    values[fixed] = exact_components(points).ravel()[fixed]
    with factorize(matrix[free][:,free]) as factor:
        values[free] = factor.solve(load[free] - matrix[free][:,fixed] @ values[fixed])
    reference = fem.Function(space)
    reference.x.array[:] = values
    difference = reference-exact
    l2 = np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(difference,difference)*dx)))
    assert l2 < 1e-10
    classical.append({"resolution":resolution,"pressure_L2":float(l2)})
classical
